# Importing Libraries

In [62]:
import requests
import json
import pandas as pd
import openpyxl
from bs4 import BeautifulSoup
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [68]:
stateCodes = pd.read_excel('DE/fresh/raktradar/data/India_State_Codes.xlsx')
stateCodes = stateCodes.iloc[2:]

stateCodes.columns = ['StateName','StateCode']

state_dict = dict(zip(stateCodes['StateName'],stateCodes['StateCode']))
print(stateCodes)

                                   StateName StateCode
2                Andaman and Nicobar Islands        35
3                             Andhra Pradesh        28
4                          Arunachal Pradesh        12
5                                      Assam        18
6                                      Bihar        10
7                                 Chandigarh        94
8                               Chhattisgarh        22
9   Dadra And Nagar Haveli And Daman And Diu        25
10                                     Delhi        97
11                                       Goa        30
12                                   Gujarat        24
13                                   Haryana        96
14                          Himachal Pradesh        92
15                         Jammu and Kashmir        91
16                                 Jharkhand        20
17                                 Karnataka        29
18                                    Kerala        32
19        

In [67]:
bloodGroup = pd.read_excel('DE/fresh/raktradar/data/Blood_Group_Codes.xlsx')
bloodGroup = bloodGroup.iloc[2:]

bloodGroup.columns = ['Blood Group','Group Code']

blood_dict = dict(zip(bloodGroup['Blood Group'],bloodGroup['Group Code']))

print(bloodGroup)

   Blood Group Group Code
2         A+Ve         11
3         A-Ve         12
4         B+Ve         13
5         B-Ve         14
6         O+Ve         15
7         O-Ve         16
8        AB+Ve         17
9        AB-Ve         18
10       Oh+VE         22
11       Oh-VE         23


In [74]:
bloodComponents = pd.read_excel('DE/fresh/raktradar/data/BloodComponents.xlsx')
bloodComponents.columns = ['Blood Component','Component Code']
blood_dict = dict(zip(bloodComponents['Blood Component'],bloodComponents['Component Code']))
print(bloodComponents)

                Blood Component  Component Code
0                   Whole Blood              11
1        Packed Red Blood Cells              12
2           Fresh Frozen Plasma              13
3         Single Donor Platelet              14
4          Platelet Rich Plasma              16
5               Cryoprecipitate              17
6           Single Donor Plasma              18
7                        Plasma              19
8          Platelet Concentrate              20
9              Cryo Poor Plasma              21
10       Random Donor Platelets              23
11  Sagm Packed Red Blood Cells              28
12               Irradiated RBC              29
13             Leukoreduced Rbc              30


In [ ]:
spark = SparkSession.builder \
    .appName('state_dist_parquet') \
    .getOrCreate()

df = spark.read.parquet('DE/fresh/raktradar/data/state_dist_codes.parquet')
print(df.count())
print(df.columns,len(df.columns))
# df.select(['State Code']).distinct().show(40)


657009
['State Code', 'District Code', 'Sub District Code', 'Town-Village Code', 'Town-Village Name'] 5


In [64]:
state_dist_list = df.groupBy('State Code').agg(
    F.collect_set('District Code').alias('districtCodeList')
)
state_dist_list.show(truncate = False)

+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|State Code|districtCodeList                                                                                                                                                                                                                                                                                                                                                        |
+----------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
url = 'https://eraktkosh.mohfw.gov.in/BLDAHIMS/bloodbank/stockAvailability.cnt'
params = {
    'hmode' : 'GETNEARBYSTOCKDETAILS',
    'stateCode' : 35,
    'districtCode' : 640,
    'bloodGroup' : 13,
    'bloodComponent' : 11,
    'lang' : 0
}
response = requests.get(url, params = params)
print(response.json())

{'data': [['1', 'Pillar Health Centre<br/>Lamba Line, Post Box No:526, Junglighat Post. PortBlair, , Port Blair, South Andaman, Andaman and Nicobar Islands<br/>Phone: 9679535438 ,Fax: -, Email: pillar.portblair@gmail.com', 'Charitable/Vol', "<p class='text-success'>Available, B+Ve:12</p>", '2026-08-09 23:39:10', 'Blood Bank'], ['2', 'G.B.Pant Hospital Atlanta Point Medical College<br/>G B Pant Hospital, Atlanta Point, Sri Vijay Puram, , Port Blair , South Andaman, Andaman and Nicobar Islands<br/>Phone: 9962794804 ,Fax: -, Email: bbgbpantportblair@gmail.com', 'Govt.', "<p class='text-danger'><b>Whole Blood</b>Not Available Search for another Component", '2026-04-27 19:24:33', 'Blood Bank']]}
